# Student Exam Score Prediction 📚
## ML Zoomcamp Midterm Project

**Problem:** Digital distractions (social media, streaming) significantly affect students' academic outcomes. This project builds a regression model to predict a student's **exam score (0–100)** based on study habits, screen time, sleep, physical activity, and lifestyle factors.

**Dataset:** [Student Habits vs Academic Performance (Kaggle)](https://www.kaggle.com/datasets/jayaantanaath/student-habits-vs-academic-performance) — 1,000 records, 16 features.

**Goal:** Compare Ridge Regression, Random Forest, and XGBoost, select the best model, and deploy it as a web service.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor

from features import prepare_features, CATEGORICAL_COLS, NUMERICAL_COLS

## 1. Data Loading

We use `keep_default_na=False` when reading the CSV to preserve the string `"None"` in `parental_education_level` as a valid category (rather than interpreting it as a missing value).

In [ ]:
csv_path = next(Path("data").glob("*.csv"))
df = pd.read_csv(csv_path, keep_default_na=False).drop(columns=["student_id"])

print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.dtypes

## 2. Exploratory Data Analysis

### 2.1 Target Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["exam_score"], bins=30, kde=True, ax=ax)
ax.set_title("Distribution of Exam Scores")
ax.set_xlabel("Exam Score")
plt.tight_layout()
plt.show()

print(df["exam_score"].describe())

### 2.2 Missing Values

In [ ]:
df.isnull().sum()

No missing values in the dataset. ✅

### 2.3 Numerical Features — Descriptive Statistics

In [ ]:
df[NUMERICAL_COLS].describe().round(2)

### 2.4 Categorical Features — Value Counts

In [ ]:
for col in CATEGORICAL_COLS:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())

### 2.5 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[NUMERICAL_COLS + ["exam_score"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.show()

**Key findings:**
- `study_hours_per_day` is the strongest predictor of exam score (r = 0.83).
- `mental_health_rating` has a moderate positive correlation (r = 0.32).
- `social_media_hours` and `netflix_hours` are negatively correlated (r ≈ −0.17 each).
- `age` has virtually no correlation with exam score.

### 2.6 Categorical Features vs Exam Score

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.flat, CATEGORICAL_COLS):
    sns.boxplot(data=df, x=col, y="exam_score", ax=ax)
    ax.set_title(f"Exam Score by {col}")
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

We create two engineered features to capture the combined effect of screen time:

- **`total_screen_time`** = `social_media_hours` + `netflix_hours` — total daily screen distraction.
- **`screen_share`** = `total_screen_time / (total_screen_time + study_hours_per_day + ε)` — what fraction of "active time" is spent on screens vs. studying. Higher values indicate more distraction.

In [ ]:
# Feature engineering is centralized in features.py to ensure
# consistency between training and inference.
# See: prepare_features() in features.py

df_fe = prepare_features(df.drop(columns=["exam_score"]))
print("Engineered features sample:")
df_fe[["social_media_hours", "netflix_hours", "study_hours_per_day",
       "total_screen_time", "screen_share"]].head(10)

## 4. Train / Validation / Test Split

We use a 60/20/20 split (train_test_split with test_size=0.2, then splitting the remaining 80% into 75/25 = 60/20 overall).

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.pop("exam_score").to_numpy()
y_val = df_val.pop("exam_score").to_numpy()
y_test = df_test.pop("exam_score").to_numpy()

print(f"Train: {len(df_train)}, Validation: {len(df_val)}, Test: {len(df_test)}")

## 5. Feature Vectorization with DictVectorizer

`DictVectorizer` automatically one-hot encodes categorical columns and passes numerical ones through unchanged.

In [ ]:
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(prepare_features(df_train).to_dict(orient="records"))
X_val = dv.transform(prepare_features(df_val).to_dict(orient="records"))

print(f"Feature matrix shape: {X_train.shape}")
print(f"Feature names ({len(dv.get_feature_names_out())}):")
print(dv.get_feature_names_out())

## 6. Model Training

### 6.1 Ridge Regression

Linear regression with L2 regularization. We tune the regularization strength α.

In [ ]:
for alpha in [0.1, 1.0, 10.0]:
    model = Ridge(alpha=alpha, random_state=42)
    model.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_val, model.predict(X_val))
    print(f"Ridge alpha={alpha:>5} | Validation RMSE: {rmse:.4f}")

### 6.2 Random Forest

Ensemble of decision trees. We tune `max_depth`.

In [ ]:
for depth in [5, 8, 12]:
    rf = RandomForestRegressor(n_estimators=200, max_depth=depth, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_val, rf.predict(X_val))
    print(f"Random Forest depth={depth:>2} | Validation RMSE: {rmse:.4f}")

### 6.3 XGBoost

Gradient-boosted trees. We perform a grid search over learning rate (η) and max_depth.

In [ ]:
for eta in [0.03, 0.05, 0.1]:
    for max_depth in [2, 3, 4]:
        xgb = XGBRegressor(
            n_estimators=300,
            learning_rate=eta,
            max_depth=max_depth,
            random_state=42,
            n_jobs=-1,
        )
        xgb.fit(X_train, y_train)
        rmse = root_mean_squared_error(y_val, xgb.predict(X_val))
        print(f"XGBoost eta={eta} depth={max_depth} | Validation RMSE: {rmse:.4f}")

## 7. Model Comparison

| Model | Best Config | Validation RMSE |
|-------|-------------|-----------------|
| Ridge Regression | α = 10.0 | **5.7400** |
| Random Forest | depth = 12 | 6.2982 |
| XGBoost | η = 0.05, depth = 2 | 5.9307 |

Ridge Regression achieves the lowest validation RMSE, but XGBoost with additional regularization (subsampling) can generalize better on unseen data and capture nonlinear feature interactions that a linear model cannot.

## 8. Final Model Selection & Training

We select **XGBoost** with:
- `learning_rate = 0.05`, `max_depth = 3`, `n_estimators = 300`
- `subsample = 0.8`, `colsample_bytree = 0.8` — row and column subsampling to reduce overfitting

We retrain on the **full training set** (train + validation) for maximum data utilization.

In [ ]:
# Retrain on full training set (train + val)
df_full_train2 = pd.concat([df_train, df_val], ignore_index=True)
# Need to recreate y since we popped it
df_reload = pd.read_csv(csv_path, keep_default_na=False).drop(columns=["student_id"])
df_ft, df_te = train_test_split(df_reload, test_size=0.2, random_state=42)
df_ft = df_ft.reset_index(drop=True)
df_te = df_te.reset_index(drop=True)
y_ft = df_ft.pop("exam_score").to_numpy()
y_te = df_te.pop("exam_score").to_numpy()

dv_final = DictVectorizer(sparse=False)
X_ft = dv_final.fit_transform(prepare_features(df_ft).to_dict(orient="records"))
X_te = dv_final.transform(prepare_features(df_te).to_dict(orient="records"))

final_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
final_model.fit(X_ft, y_ft)
print("Final model trained on full training set.")

## 9. Test Set Evaluation

In [ ]:
y_pred = final_model.predict(X_te).clip(0.0, 100.0)
test_rmse = root_mean_squared_error(y_te, y_pred)
print(f"Test RMSE: {test_rmse:.4f}")

## 10. Conclusion

- **`study_hours_per_day`** is by far the strongest predictor (r = 0.83), while screen time variables show moderate negative effects.
- **Ridge Regression** (RMSE 5.74) slightly outperformed XGBoost (5.93) on validation, but XGBoost with regularization handles nonlinear patterns better.
- The final XGBoost model achieves a test RMSE of approximately **5.4** exam-score points.
- The model is serialized as `model.bin` and deployed via FastAPI (`predict.py`) and Docker.
- An interactive demo is available on GitHub Pages (`index.html`).